# Auditoria RAG - CORADIR Movilidad Electrica

Notebook para mostrar el proceso completo de trabajo sobre datos, corpus, retrieval y generacion.

La idea no es demostrar solo que el chatbot responde, sino explicar como se transformo la documentacion, como se midio si recuperaba los chunks correctos y que decisiones tecnicas se tomaron a partir de esas metricas.

**Este notebook no llama al LLM.** Trabaja sobre archivos ya generados por el proyecto para que la evidencia sea reproducible y rapida de revisar.

## 0. Que evidencia se va a revisar

El recorrido del notebook es:

1. Cargar corpus RAG generado desde la documentacion original.
2. Medir calidad del corpus: cantidad de documentos, longitud, TTR/MATTR y outliers.
3. Cargar benchmark extendido y ver resultado final.
4. Auditar retrieval: pregunta, fuentes esperadas, chunks recuperados, ranking, scores y contenido.
5. Medir si el sistema recupera el chunk correcto con Precision@5, Recall@5, MRR y Top-1.
6. Medir generacion con Token Overlap y Context Faithfulness.
7. Traducir metricas a decisiones tecnicas concretas.

In [ ]:
from pathlib import Path
import json
import re
import unicodedata
from collections import Counter, defaultdict

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_colwidth', 140)
plt.style.use('default')

# Resolver raiz del repo tanto si el notebook se abre desde la raiz como desde docs/.
CWD = Path.cwd()
if (CWD / 'dataset' / 'knowledge_base_movilidad.jsonl').exists():
    ROOT = CWD
elif (CWD.parent / 'dataset' / 'knowledge_base_movilidad.jsonl').exists():
    ROOT = CWD.parent
else:
    ROOT = CWD

PATHS = {
    'corpus': ROOT / 'dataset' / 'knowledge_base_movilidad.jsonl',
    'benchmark_final': ROOT / 'docs' / 'benchmark_extendido_strict_post_extraccion_v3.json',
    'audit_json': ROOT / 'docs' / 'auditoria_rag_chunks_clase6.json',
    'audit_csv': ROOT / 'docs' / 'auditoria_rag_chunks_clase6.csv',
    'generation_metrics': ROOT / 'docs' / 'evaluacion_metricas_generacion_clase6.json',
    'eda_report': ROOT / 'docs' / 'eda_corpus_rag.md',
}

for name, path in PATHS.items():
    print(f'{name:18} -> {path} | exists={path.exists()}')

## 1. Carga del corpus RAG

El sistema no indexa el JSON administrativo completo como un bloque. Primero lo transforma en documentos mas chicos y trazables en `dataset/knowledge_base_movilidad.jsonl`.

Cada fila conserva:

- `section`: area del dominio, por ejemplo precios, vehiculos, agencias.
- `title`: titulo humano del documento.
- `source_path`: ruta logica dentro del JSON original.
- `content`: texto que puede ser recuperado por el RAG.

In [ ]:
def load_jsonl(path: Path):
    rows = []
    for line in path.read_text(encoding='utf-8').splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows

corpus = load_jsonl(PATHS['corpus'])
df_corpus = pd.DataFrame(corpus)
print('Documentos RAG:', len(df_corpus))
df_corpus[['section', 'title', 'source_path', 'content']].head(8)

## 2. EDA del corpus

Antes de cambiar chunking, normalizacion o lematizacion, se mide el corpus. Esto evita hacer transformaciones por intuicion.

Metricas usadas:

- cantidad de documentos por seccion
- tokens por documento
- TTR: diversidad lexica simple
- MATTR: diversidad lexica con ventana movil
- outliers: documentos demasiado cortos o largos

In [ ]:
STOP = {
    'de','del','la','las','el','los','un','una','y','o','a','en','por','con','para','que',
    'como','cual','cuales','cuanto','cuanta','tiene','tienen','es','son','se','si','no','al',
    'lo','su','sus','este','esta','estos','estas','pregunta','respuesta'
}

def normalize_text(text):
    text = str(text).lower()
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r'[^a-z0-9]+', ' ', text)
    return ' '.join(text.split())

def tokens(text):
    return [tok for tok in normalize_text(text).split() if len(tok) > 1 and tok not in STOP]

def ttr(tok):
    return len(set(tok)) / len(tok) if tok else 0

def mattr(tok, window=50):
    if not tok:
        return 0
    if len(tok) <= window:
        return ttr(tok)
    scores = []
    for i in range(0, len(tok) - window + 1):
        scores.append(ttr(tok[i:i+window]))
    return sum(scores) / len(scores)

df_corpus['tokens'] = df_corpus['content'].map(tokens)
df_corpus['token_count'] = df_corpus['tokens'].map(len)
df_corpus['ttr'] = df_corpus['tokens'].map(ttr)
df_corpus['mattr'] = df_corpus['tokens'].map(mattr)

summary = {
    'documentos': len(df_corpus),
    'tokens_totales': int(df_corpus['token_count'].sum()),
    'tokens_promedio': round(df_corpus['token_count'].mean(), 2),
    'tokens_mediana': round(df_corpus['token_count'].median(), 2),
    'tokens_p90': round(df_corpus['token_count'].quantile(0.9), 2),
    'mattr_promedio': round(df_corpus['mattr'].mean(), 4),
}
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

df_corpus['token_count'].plot(kind='hist', bins=25, ax=axes[0], color='#2563eb', edgecolor='white')
axes[0].set_title('Distribucion de tokens por chunk')
axes[0].set_xlabel('Tokens')

section_counts = df_corpus['section'].value_counts().head(12).sort_values()
section_counts.plot(kind='barh', ax=axes[1], color='#0f766e')
axes[1].set_title('Documentos por seccion')
axes[1].set_xlabel('Cantidad')

plt.tight_layout()
plt.show()

In [ ]:
# Chunks extremos para justificar decisiones de revision.
outliers = pd.concat([
    df_corpus.nsmallest(8, 'token_count'),
    df_corpus.nlargest(8, 'token_count')
])
outliers[['section', 'title', 'source_path', 'token_count', 'mattr']]

### Decision a partir del EDA

El corpus tiene documentos cortos y algunos documentos largos. La decision tomada fue **no aplicar lematizacion global** porque podia borrar terminos exactos importantes del dominio, como modelos, precios, versiones, provincias, telefonos o codigos.

La mejora priorizada fue:

- mantener metadata trazable (`section`, `title`, `source_path`)
- medir retrieval por fuentes esperadas
- auditar chunks recuperados antes de cambiar otra vez el corpus

## 3. Benchmark extendido

El benchmark extendido agrega casos mas exigentes y permite medir si el sistema encuentra informacion puntual de vehiculos, precios, agencias, compra, carga y empresa.

Aca la idea es separar dos preguntas:

1. La respuesta final contiene las keywords esperadas?
2. El retrieval trajo los chunks que correspondian?

In [ ]:
benchmark = json.loads(PATHS['benchmark_final'].read_text(encoding='utf-8'))
summary_benchmark = benchmark['summary']
summary_benchmark

In [ ]:
df_benchmark = pd.DataFrame(benchmark['results'])
print('Casos:', len(df_benchmark))
print('Accuracy por keywords:', df_benchmark['passed'].mean())
df_benchmark[['id', 'category', 'question', 'expected_keywords', 'passed', 'response']].head(8)

## 4. Auditoria de chunks recuperados

Este es el punto central para defender el retrieval.

El archivo `auditoria_rag_chunks_clase6.json` recompone, para cada pregunta:

- fuentes esperadas (`expected_sources`)
- chunks recuperados
- ranking del chunk
- modo de recuperacion (`keyword` o `vector`)
- score lexical o ranking vectorial
- contenido completo del chunk
- tokens de pregunta, keywords y respuesta
- overlap entre chunk y pregunta/keywords/respuesta
- diagnostico por caso

In [ ]:
audit = json.loads(PATHS['audit_json'].read_text(encoding='utf-8'))
df_chunks = pd.read_csv(PATHS['audit_csv'])
audit['summary']

In [ ]:
metrics = audit['summary']
retrieval_kpis = pd.DataFrame([
    {'metrica': 'Todas las fuentes esperadas en top 5', 'valor': metrics['cases_with_all_expected_sources_in_top_k'], 'total': metrics['total_cases']},
    {'metrica': 'Alguna fuente esperada en top 5', 'valor': metrics['cases_with_any_expected_source_in_top_k'], 'total': metrics['total_cases']},
    {'metrica': 'Fuente esperada en top 1', 'valor': metrics['cases_with_expected_source_at_top_1'], 'total': metrics['total_cases']},
])
retrieval_kpis['porcentaje'] = (retrieval_kpis['valor'] / retrieval_kpis['total'] * 100).round(1)
retrieval_kpis

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(retrieval_kpis['metrica'], retrieval_kpis['porcentaje'], color=['#0f766e', '#16a34a', '#0284c7'])
ax.set_ylim(0, 105)
ax.set_ylabel('% de casos')
ax.set_title('Auditoria de retrieval por fuentes esperadas')
for i, row in retrieval_kpis.iterrows():
    ax.text(i, row['porcentaje'] + 2, f"{int(row['valor'])}/{int(row['total'])}", ha='center', fontweight='bold')
plt.xticks(rotation=18, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Metricas globales de retrieval desde la auditoria final.
pd.DataFrame([
    {'metrica': 'Precision@5 promedio', 'valor': metrics['mean_precision_at_k']},
    {'metrica': 'Recall@5 promedio', 'valor': metrics['mean_recall_at_k']},
    {'metrica': 'Primer rank relevante promedio', 'valor': metrics['mean_first_relevant_rank']},
    {'metrica': 'Soporte de respuesta en chunks esperados', 'valor': metrics['mean_response_support_on_expected_chunks']},
])

### Lectura de las metricas de retrieval

- **Recall@5 alto**: el sistema encuentra la evidencia en la mayoria de los casos.
- **Precision@5 baja**: tambien trae ruido; hay chunks que no son la fuente esperada.
- **Top-1 alto**: en la mayoria de los casos el chunk correcto aparece primero.
- **Casos `ranking_review`**: la fuente correcta aparece, pero no en primer lugar.
- **Casos `missing_expected_source`**: falta revisar metadata, expected source o cobertura del corpus.

In [ ]:
# Diagnostico global de casos.
diagnosis = pd.Series(audit['summary']['diagnosis_counts']).sort_values(ascending=False)
diagnosis

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
diagnosis.plot(kind='bar', ax=ax, color=['#f97316', '#0f766e', '#eab308', '#ef4444'])
ax.set_title('Diagnostico de retrieval por caso')
ax.set_ylabel('Casos')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## 5. Inspeccion caso por caso

Esta celda permite elegir un caso y ver exactamente que chunks recupero el sistema.

Campos importantes:

- `matches_expected_source`: si el chunk coincide con la fuente esperada.
- `expected_keyword_overlap`: si el chunk contiene las keywords esperadas.
- `response_token_support`: si el chunk respalda lexicalmente la respuesta.
- `content_excerpt`: extracto del chunk recuperado.

In [ ]:
# Cambiar este ID para revisar otros casos.
case_id = 'chiki_autonomia_velocidad'

case = next(c for c in audit['cases'] if c['id'] == case_id)
print('Pregunta:', case['question'])
print('Expected sources:', case['expected_sources'])
print('Respuesta:', case['response'])
print('Diagnostico:', case['diagnosis'])
print('Ranks relevantes:', case['relevant_chunk_ranks'])

rows = []
for ch in case['chunks']:
    rows.append({
        'rank': ch['rank'],
        'ok_fuente': ch['matches_expected_source'],
        'modo': ch['retrieval_mode'],
        'keyword_score': ch['keyword_score'],
        'vector_rank': ch['vector_rank'],
        'source_path': ch['source_path'],
        'kw_overlap': ch['expected_keyword_overlap'],
        'resp_support': ch['response_token_support'],
        'kw_hits': ', '.join(ch['expected_keyword_hits']),
        'extracto': ch['content_excerpt'],
    })

pd.DataFrame(rows)

In [ ]:
# Ver el contenido completo del primer chunk relevante del caso elegido.
relevant = [ch for ch in case['chunks'] if ch['matches_expected_source']]
if relevant:
    first = relevant[0]
    print('Chunk relevante rank:', first['rank'])
    print('Source:', first['source_path'])
    print('\n--- CONTENIDO COMPLETO ---')
    print(first['corpus_documents_matched'][0]['content'])
else:
    print('No se encontro chunk que matchee expected_sources para este caso.')

## 6. Metricas de generacion - Clase 6

Luego de auditar retrieval, se mide la respuesta generada.

Metricas:

- **Token Overlap**: tokens de la respuesta contra keywords esperadas.
- **Context Faithfulness**: tokens de la respuesta contra el contexto recuperado.

Esto permite detectar respuestas que pasan por keywords pero agregan informacion no trazada al contexto.

In [ ]:
gen_metrics = json.loads(PATHS['generation_metrics'].read_text(encoding='utf-8'))
gen_metrics['summary']

In [ ]:
df_gen = pd.DataFrame(gen_metrics['rows'])
df_gen[['id', 'category', 'token_overlap', 'context_faithfulness', 'diagnosis']].head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

df_gen['token_overlap'].plot(kind='hist', bins=10, ax=axes[0], color='#0f766e', edgecolor='white')
axes[0].set_title('Token Overlap')
axes[0].set_xlabel('Score')

df_gen['context_faithfulness'].plot(kind='hist', bins=10, ax=axes[1], color='#0284c7', edgecolor='white')
axes[1].set_title('Context Faithfulness')
axes[1].set_xlabel('Score')

plt.tight_layout()
plt.show()

In [ ]:
# Casos con menor faithfulness para revision manual o ajuste de capa extractiva.
df_gen.sort_values('context_faithfulness').head(12)[[
    'id', 'category', 'token_overlap', 'context_faithfulness', 'diagnosis', 'unsupported_response_tokens'
]]

## 7. Como se tradujeron metricas en decisiones

Esta tabla resume el razonamiento usado para ajustar y justificar el sistema.

In [ ]:
decisiones = pd.DataFrame([
    {
        'senal_observada': 'JSON original jerarquico y heterogeneo',
        'metrica_o_evidencia': 'Documentos grandes o mezclados; riesgo de recuperar bloques amplios',
        'decision': 'Transformar a JSONL con chunks mas chicos por FAQ, precio, ficha, agencia o condicion',
    },
    {
        'senal_observada': 'MATTR promedio alto y terminos comerciales importantes',
        'metrica_o_evidencia': 'MATTR 0.8538; modelos, precios, telefonos y provincias son terminos exactos',
        'decision': 'No aplicar lematizacion global; priorizar metadata y trazabilidad',
    },
    {
        'senal_observada': 'Benchmark extendido inicial bajo',
        'metrica_o_evidencia': '35/64 por keywords',
        'decision': 'Separar fallas de retrieval y fallas de respuesta; agregar expected_sources',
    },
    {
        'senal_observada': 'Recall@5 alto pero Precision@5 baja',
        'metrica_o_evidencia': 'Recall@5 96.9%; Precision@5 31.9%',
        'decision': 'El dato suele estar; el problema principal es ruido/ranking. Mantener hibrido y revisar filtros/ranking',
    },
    {
        'senal_observada': 'Fuente esperada no siempre aparece primera',
        'metrica_o_evidencia': '56/64 con fuente esperada en top 1; 7 casos ranking_review',
        'decision': 'Marcar casos para mejorar ranking antes de cambiar modelo',
    },
    {
        'senal_observada': 'Respuesta correcta pero contexto no siempre trazado',
        'metrica_o_evidencia': 'Context Faithfulness promedio 0.8949; 13 casos a revisar',
        'decision': 'Revisar capa extractiva para evitar informacion adicional o no trazada',
    },
    {
        'senal_observada': 'Preguntas fuera de dominio pueden disparar contexto irrelevante',
        'metrica_o_evidencia': 'Dataset no respondible y guardrail',
        'decision': 'Rechazar antes de llamar al LLM cuando no hay dato o no corresponde al dominio',
    },
])
decisiones

## 8. Conclusiones defendibles

1. La mejora principal fue de **datos y retrieval**, no de cambio de modelo.
2. El JSON original se transformo en un corpus RAG con 111 documentos trazables.
3. El EDA justifico no aplicar normalizacion agresiva ni lematizacion global.
4. El benchmark extendido permitio medir fuentes esperadas, no solo respuestas finales.
5. La auditoria de chunks muestra que el sistema recupera todas las fuentes esperadas en 61/64 casos y alguna fuente esperada en 63/64.
6. La Precision@5 baja indica ruido en contexto; por eso la mejora recomendada es ranking/filtros, no cambiar primero el LLM.
7. Context Faithfulness marca casos donde la respuesta es correcta por keywords pero requiere revision de trazabilidad.

La defensa tecnica es: **mejoramos la respuesta porque antes mejoramos la documentacion, el corpus, la trazabilidad y la medicion del retrieval**.